In [3]:
from pathlib import Path
import duckdb

database_path = Path("../olist_sales_intelligence.duckdb").resolve()
con = duckdb.connect(str(database_path), read_only=True)

portfolio_kpis = con.execute("""
    SELECT
        COUNT(*) AS total_customers,
        ROUND(SUM(total_revenue), 2) AS total_revenue,
        ROUND(AVG(total_orders), 2) AS average_orders_per_customer,
        ROUND(AVG(average_order_value), 2) AS average_order_value,
        ROUND(AVG(average_review_score), 2) AS average_review_score,
        SUM(CASE WHEN risk_level = 'High' THEN 1 ELSE 0 END) AS high_risk_customers,
        ROUND(
            100.0 * SUM(CASE WHEN risk_level = 'High' THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS high_risk_customer_rate_pct
    FROM customer_360
""").df()

portfolio_kpis

segment_performance = con.execute("""
    SELECT
        customer_segment,
        COUNT(*) AS customers,
        ROUND(SUM(total_revenue), 2) AS revenue,
        ROUND(100.0 * SUM(total_revenue) / SUM(SUM(total_revenue)) OVER (), 2)
            AS revenue_share_pct,
        ROUND(AVG(average_order_value), 2) AS average_order_value,
        ROUND(AVG(average_review_score), 2) AS average_review_score,
        ROUND(100.0 * AVG(late_order_rate), 2) AS late_delivery_rate_pct
    FROM customer_360
    GROUP BY customer_segment
    ORDER BY revenue DESC
""").df()

segment_performance

,customer_segment,customers,revenue,revenue_share_pct,average_order_value,average_review_score,late_delivery_rate_pct
0,High Value,18151,4892380.15,31.73,249.81,4.08,9.66
1,Inactive,23325,3194370.14,20.72,136.80,4.15,7.49
2,At Risk,17440,2875774.65,18.65,159.90,4.22,4.45
3,New,17154,2250332.36,14.59,131.18,4.30,6.55
4,Active,17288,2206916.45,14.31,127.25,4.02,12.79
